# FaceUp — train emotion models on Colab GPU

Trains our own classifiers (softmax baseline + custom CNN) on FER2013, remapped to the app's 5 classes. Runs the repo's real `ml/` code on a GPU.

**Steps:**
1. **Runtime → Change runtime type → GPU**, then run cells top-to-bottom.
2. When asked, upload your `kaggle.json` (Kaggle → account → Settings → API → *Create New Token*) to fetch FER2013.
3. At the end, `artifacts.zip` downloads — unzip its contents into `ml/artifacts/` in the local repo so the app can use the model.

In [ ]:
# Match the app's Keras dialect so the saved .keras loads in the app later.
%env TF_USE_LEGACY_KERAS=1
!pip install -q tf-keras
import tensorflow as tf
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
!git clone -q https://github.com/Explorer0l/FaceUp.git
%cd FaceUp
!git checkout -q feature/ml-training
!pip install -q opencv-python-headless scikit-learn matplotlib

## Get FER2013 (via Kaggle)
Alternative: mount Google Drive and set `%env FACEUP_DATA_ROOT=/content/drive/MyDrive/your/fer2013` instead of the cell below.

In [ ]:
from google.colab import files
print('Upload kaggle.json:'); files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d msambare/fer2013
!mkdir -p data/fer2013 && unzip -q -o fer2013.zip -d data/fer2013
!ls data/fer2013

In [ ]:
# Topic 7 baseline + Topic 8 centerpiece (Keras .fit)
!python -m ml.train --model softmax --epochs 20
!python -m ml.train --model cnn --epochs 40

In [ ]:
# Optional — the same CNN via an explicit tf.GradientTape loop (Topic 8)
!python -m ml.train --model cnn --loop gradtape --epochs 15

In [ ]:
!python -m ml.evaluate --model softmax
!python -m ml.evaluate --model cnn

In [ ]:
# Download trained weights + figures, then unzip into ml/artifacts/ locally.
!zip -qr artifacts.zip ml/artifacts
from google.colab import files; files.download('artifacts.zip')